In [15]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/accepted-2007/accepted_2007_to_2018Q4.csv


In [16]:
import kagglehub
import pandas as pd
from kagglehub import KaggleDatasetAdapter

# Set the correct file name from the dataset
file_path = "accepted_2007_to_2018Q4.csv"

# Load the dataset using kagglehub
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "seegumanasa/accepted-2007",  # Kaggle dataset name
    file_path                     # CSV file path within the dataset
)

# Convert 'issue_d' to datetime format (if it exists)
df['issue_d'] = pd.to_datetime(df['issue_d'], format="%b-%Y", errors='coerce')

# Filter the dataset to include only loans from 2018 and newer
df_filtered = df[df['issue_d'].dt.year >= 2018]

# Display summary of the filtered dataset
print("Filtered dataset shape:", df_filtered.shape)
print("Sample rows:\n", df_filtered.head())

/tmp/ipykernel_35/1745379898.py:9: DeprecationWarning: load_dataset is deprecated and will be removed in a future version.
  df = kagglehub.load_dataset(
/usr/local/lib/python3.11/dist-packages/kagglehub/pandas_datasets.py:91: DtypeWarning: Columns (0,19,49,59,118,129,130,131,134,135,136,139,145,146,147) have mixed types. Specify dtype option on import or set low_memory=False.
  result = read_function(


Filtered dataset shape: (495242, 151)
Sample rows:
                id  member_id  loan_amnt  funded_amnt  funded_amnt_inv  \
421097  130954621        NaN     5000.0       5000.0           5000.0   
421098  130964697        NaN    15000.0      15000.0          15000.0   
421099  130955326        NaN    11200.0      11200.0          11200.0   
421100  130504052        NaN    25000.0      25000.0          25000.0   
421101  130956066        NaN     3000.0       3000.0           3000.0   

              term  int_rate  installment grade sub_grade  ...  \
421097   36 months     20.39       186.82     D        D4  ...   
421098   36 months      9.92       483.45     B        B2  ...   
421099   60 months     30.79       367.82     G        G1  ...   
421100   60 months     21.85       688.35     D        D5  ...   
421101   36 months      7.34        93.10     A        A4  ...   

       hardship_payoff_balance_amount hardship_last_payment_amount  \
421097                            NaN     

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


In [17]:
columns_to_drop = ['id', 'member_id', 'emp_title', 'zip_code', 'addr_state', 'url']
df_filtered = df_filtered.drop(columns=columns_to_drop, errors='ignore')


In [18]:
df_filtered = df_filtered.dropna(subset=['loan_status', 'annual_inc', 'dti'])


In [19]:
df_filtered['income_tier'] = pd.cut(
    df_filtered['annual_inc'],
    bins=[0, 40000, 80000, float('inf')],
    labels=['Low', 'Mid', 'High']
)


In [20]:
df_filtered['rir'] = df_filtered['total_pymnt'] / df_filtered['annual_inc']


In [21]:
df_filtered['high_risk'] = (
    (df_filtered['dti'] > 35) &
    (df_filtered['annual_inc'] < 40000) &
    (df_filtered['loan_status'] == 'Charged Off')
)


In [22]:
df_filtered['purpose_grouped'] = df_filtered['purpose'].replace({
    'car': 'auto', 
    'credit_card': 'debt', 
    'debt_consolidation': 'debt', 
    'home_improvement': 'home', 
    'major_purchase': 'other',
    'medical': 'other', 
    'vacation': 'other', 
    'renewable_energy': 'other',
})


In [23]:
df_filtered.to_csv("cleaned_lendingclub_2018plus.csv", index=False)


In [24]:
# Select only the important fields you want to keep
selected_columns = [
    'loan_status',
    'purpose',
    'income_tier',
    'dti',
    'grade',
    'high_risk',
    'annual_inc',
    'total_pymnt',
    'issue_d'  # optional for time filters
]

df_selected = df_filtered[selected_columns]

# Export new cleaned CSV
df_selected.to_csv("loan_dashboard_ready.csv", index=False)
